<a href="https://colab.research.google.com/github/SAWGraph/public/blob/main/UseCases/UC1-Testing/UC1-CQ3b%265/UC1_CQ3b%2C5_query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# @title Install Packages
%%capture
!pip install mapclassify
!pip install SPARQLWrapper
!pip install rdflib

In [2]:
# @title Import packages and define functions
from branca.element import Figure                                   # For controlling the size of the final map
import folium                                                       # For map layer control
from folium.plugins import StripePattern                            # For filling polygons with a striped pattern
import geopandas as gpd                                             # For geospatial dataframes
import pandas as pd                                                 # For dataframes
from shapely import wkt                                             # For working with WKT coordinates in a GeoDataFrame
from shapely.ops import transform                                   # To fix well coordinates (swap from (y, x) to (x, y)) - can be removed in the future once the data are fixed in FRINK
from SPARQLWrapper import SPARQLWrapper2, JSON, GET, POST, BASIC    # For querying SPARQL endpoints
import rdflib                                                       # For working with URIs
from datetime import date                                           # For adding the date to saved map files

def convertToDataframe(results):
  d = []
  for x in results.bindings:
        row = {}
        for k in x:
            v = x[k]
            vv = rdflib.term.Literal(v.value, datatype=v.datatype).toPython()  # type: ignore[no-untyped-call]
            row[k] = vv
        d.append(row)
  df = pd.DataFrame(d)
  return df

# Question: What water wells in Maine (or a Maine county) are potentially hydrologically connected to PFAS contaminated private water supply wells where results have been at least *x* ng/L?

In [3]:
# @title Select administrative region
admin_region = "23011 (Kennebec County, Maine)" # @param [ "23 (Maine)", "23001 (Androscoggin County, Maine)", "23005 (Cumberland County, Maine)", "23011 (Kennebec County, Maine)", "23019 (Penboscot County, Maine)", "23031 (York County, Maine)" ] {"allow-input":true}

regionCode = admin_region.split()[0]
if regionCode == 'All':
  regionFilter = ''
else:
  if len(regionCode) <= 5:
    regionURI = 'kwgr:administrativeRegion.USA.' + regionCode
    regionFilter = '''?csubd rdf:type kwg-ont:AdministrativeRegion_3 ;
            kwg-ont:administrativePartOf+ ''' + regionURI + ''' . '''
  else:
    # need to implement region filters with datacommons geoids.
    regionFilter = ''
# print(regionURI)
print(regionFilter)

?csubd rdf:type kwg-ont:AdministrativeRegion_3 ;
            kwg-ont:administrativePartOf+ kwgr:administrativeRegion.USA.23011 . 


In [4]:
# @title Select PFAS type
pfas_choice = "PFOA (perfluorooctanoic acid)" # @param ['PFOA (perfluorooctanoic acid)', 'PFOS (perfluorooctane sulfonic acid)', 'PFHxS (perfluorohexane sulfonic acid)', 'PFNA (perfluorononanoic acid)', 'PFDA (perfluorodecanoic acid)', 'PFHxA (hexafluoropropylene oxide dimer acid (GenX))'] {"allow-input":true}
pfas_type = pfas_choice.split()[0]
start = pfas_choice.find('(') + 1
end = pfas_choice.find(')')
pfas_name = pfas_choice[start:end].capitalize()
print(pfas_type)
print(pfas_name)

PFOA
Perfluorooctanoic acid


In [5]:
# @title Select minimum contamination level
min_contamination = 100 # @param {"allow-input":true}
print(min_contamination)

100


# Query

In [6]:
# @title Retrieve aquifers, s2 cells, and sample results
me_egad_sample_point_type = "PWSW" # private water supply well
unit = 'NanoGM-PER-L'
query = f"""
PREFIX coso: <http://w3id.org/coso/v1/contaminoso#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX gwml2: <http://gwml2.org/def/gwml2#>
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX kwgr: <http://stko-kwg.geog.ucsb.edu/lod/resource/>
PREFIX me_mgs: <http://sawgraph.spatialai.org/v1/me-mgs#>
PREFIX me_egad: <http://w3id.org/sawgraph/v1/me-egad#>
PREFIX me_egad_data: <http://w3id.org/sawgraph/v1/me-egad-data#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX spatial: <http://purl.org/spatialai/spatial/spatial-full#>
PREFIX unit: <http://qudt.org/vocab/unit/>

SELECT DISTINCT ?sp ?sp_wkt ?result ?aq ?aq_wkt ?well ?well_label ?well_wkt
WHERE {{
  ?sp rdf:type coso:SamplePoint ;
      me_egad:samplePointType ?sp_type ;
      spatial:connectedTo ?sp_s2 ;
      geo:hasGeometry/geo:asWKT ?sp_wkt .
  ?sp_s2 rdf:type kwg-ont:S2Cell_Level13 ;
         spatial:connectedTo ?csubd .
  {regionFilter}

  ?obs rdf:type coso:ContaminantObservation ;
       coso:observedAtSamplePoint ?sp ;
       coso:ofSubstance ?substance ;
       coso:hasResult/coso:measurementValue ?result ;
       coso:hasResult/coso:measurementUnit ?unit .
  ?substance rdf:type <http://w3id.org/DSSTox/v1/ChemicalEntity> ;
             skos:altLabel ?substance_label .

  ?aq rdf:type gwml2:GW_Aquifer ;
      spatial:connectedTo ?sp_s2 ;
      spatial:connectedTo ?aq_s2 ;
      geo:hasGeometry/geo:asWKT ?aq_wkt .
  ?aq_s2 rdf:type kwg-ont:S2Cell_Level13 .

  ?well rdf:type me_mgs:MGS-Well ;
        spatial:connectedTo ?aq_s2 ;
        rdfs:label ?well_label ;
        geo:hasGeometry/geo:asWKT ?well_wkt .
#   OPTIONAL {{ ?well me_mgs:hasUse ?well_useiri }}
#   OPTIONAL {{ ?well me_mgs:ofWellType ?well_typeiri }}
#   OPTIONAL {{ ?well me_mgs:wellDepth/qudt:numericValue ?well_depthft }}
#   OPTIONAL {{ ?well me_mgs:wellOverburden/qudt:numericValue ?well_overburdenft }}
  VALUES ?sp_type {{ me_egad:featureType.{me_egad_sample_point_type} }}
  VALUES ?substance_label {{ "{pfas_type}" }}
  VALUES ?unit {{ unit:NanoGM-PER-L }}
  FILTER(?result > {min_contamination})
}}
"""

print(query)


PREFIX coso: <http://w3id.org/coso/v1/contaminoso#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX gwml2: <http://gwml2.org/def/gwml2#>
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX kwgr: <http://stko-kwg.geog.ucsb.edu/lod/resource/>
PREFIX me_mgs: <http://sawgraph.spatialai.org/v1/me-mgs#>
PREFIX me_egad: <http://w3id.org/sawgraph/v1/me-egad#>
PREFIX me_egad_data: <http://w3id.org/sawgraph/v1/me-egad-data#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX spatial: <http://purl.org/spatialai/spatial/spatial-full#>
PREFIX unit: <http://qudt.org/vocab/unit/>

SELECT DISTINCT ?sp ?sp_wkt ?result ?aq ?aq_wkt ?well ?well_label ?well_wkt
WHERE {
  ?sp rdf:type coso:SamplePoint ;
      me_egad:samplePointType ?sp_type ;
      spatial:connectedTo ?sp_s2 ;

In [7]:
%%time
endpoint = 'https://apps.okn.us/federation/sparql'

sparql = SPARQLWrapper2(endpoint)
sparql.setMethod(POST)
sparql.setReturnFormat(JSON)
sparql.setHTTPAuth(BASIC)
# sparql.setCredentials("USERNAME", "PASSWORD")

sparql.setQuery(query)
query_result = sparql.query()
result = convertToDataframe(query_result)
result.tail()
if result.empty:
    raise Exception("No results returned for chosen input parameters.")

CPU times: user 640 ms, sys: 241 ms, total: 881 ms
Wall time: 1min 51s


# Prep query results for mapping

In [8]:
# @title Create thematic dataframes
aquifers = result[['aq', 'aq_wkt']].copy()
aquifers.drop_duplicates(inplace=True)
aquifers['aq_wkt'] = aquifers['aq_wkt'].apply(wkt.loads)

# wells = result[['well', 'well_label', 'well_useiri', 'well_typeiri', 'well_depthft', 'well_overburdenft', 'well_wkt']].copy()
wells = result[['well', 'well_label', 'well_wkt']].copy()
wells.drop_duplicates(inplace=True)
wells['well_wkt'] = wells['well_wkt'].apply(wkt.loads)

samplepts = result[['sp', 'result', 'sp_wkt']].copy()
samplepts.drop_duplicates(inplace=True)
samplepts['sp_wkt'] = samplepts['sp_wkt'].apply(wkt.loads)

In [9]:
# @title Create GeoDataFrames
%%capture
aquifers = gpd.GeoDataFrame(aquifers, geometry='aq_wkt')
aquifers.set_crs(epsg=4326, inplace=True, allow_override=True)

wells = gpd.GeoDataFrame(wells, geometry='well_wkt')
wells.set_crs(epsg=4326, inplace=True, allow_override=True)
# wells['wellwkt'] = wells['wellwkt'].apply(lambda geom: transform(lambda x, y: (y, x), geom))

samplepts = gpd.GeoDataFrame(samplepts, geometry='sp_wkt')
samplepts.set_crs(epsg=4326, inplace=True, allow_override=True)

In [10]:
# @title Create map layers
%%capture
samplept_color = 'red'
well_color = 'deepskyblue'
aquifer_color = 'blue'
boundweight = 5

map = folium.Map(tiles=None)
folium.TileLayer('openstreetmap', name='OpenStreetMap', show=True, referrerPolicy='strict-origin-when-cross-origin').add_to(map)
folium.TileLayer('CartoDB voyager', name='Carto Voyager', show=False).add_to(map)

sp = StripePattern(angle=-30, color=aquifer_color, space_color='white', space_opacity=0.75).add_to(map)

aquifers.explore(m=map,
                 color=aquifer_color,
                 style_kwds=dict(weight=0.5*boundweight, style_function=lambda x :{'fillPattern': sp}),
                 popup=['aq'],
                 tooltip=False,
                 name=f'<span style="color: {aquifer_color};">Aquifers</span>',
                 show=True)
wells.explore(m=map,
              color=well_color,
              style_kwds=dict(weight=2*boundweight),
            #   popup=['well_label', 'well_typeiri', 'well_useiri', 'well_depthft', 'well_overburdenft', 'well'],
              popup=['well_label', 'well'],
              tooltip=False,
              name=f'<span style="color: {well_color};">Potentially connected water wells</span>',
              show=True)
samplepts.explore(m=map,
                  color=samplept_color,
                  style_kwds=dict(weight=2*boundweight),
                  popup=['result', 'sp'],
                  tooltip=False,
                  name=f'<span style="color: {samplept_color};">Private well sample points</span>',
                  show=True)

# Show map

In [11]:
# @title Display map
print()
print(f"Wells (light blue) potentially hydrologically connected to aquifers (dark blue) that are potentially contaminated at overlapping sample points (red).")
print()
print(f'Sample points are within {admin_region} and have at least one {pfas_choice} observation above {min_contamination} ng/L.')
print()
print('Use the checkboxes on the right to turn layers on/off.')
print('Left-click and drag to pan the map. Use the mouse wheel or +/- buttons on the left to zoom in/out.')
print()
folium.FitOverlays().add_to(map)
folium.LayerControl(collapsed=False).add_to(map)
fig = Figure(width='90%', height=700)
fig.add_child(map)


Wells (light blue) potentially hydrologically connected to aquifers (dark blue) that are potentially contaminated at overlapping sample points (red).

Sample points are within 23011 (Kennebec County, Maine) and have at least one PFOA (perfluorooctanoic acid) observation above 100 ng/L.

Use the checkboxes on the right to turn layers on/off.
Left-click and drag to pan the map. Use the mouse wheel or +/- buttons on the left to zoom in/out.



In [12]:
# @title Save .html copy of map
today = date.today()
fig.save(f'UC1-CQ3b,5_Water-wells-connected-PWSW-at-least-{min_contamination}-ng-per-L-{pfas_type}_{regionCode}_{today}.html')

# Looking Further

Once the results of the primary query are available, it is possible to query for additional information related to specific results.

Here you can view tables of average test results (by date and type of PFAS) for some different sample points. Select a sample point from the drop down and then press the 'play' button on the left. Different sample points can have very different sample histories.

In [13]:
sample_point = 'samplePoint.152643' #@param ['samplePoint.143624', 'samplePoint.147579', 'samplePoint.152643', 'samplePoint.154977']
samples_query = f"""
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX coso: <http://w3id.org/coso/v1/contaminoso#>
PREFIX me_egad: <http://w3id.org/sawgraph/v1/me-egad#>
PREFIX me_egad_data: <http://w3id.org/sawgraph/v1/me-egad-data#>
PREFIX qudt: <http://qudt.org/schema/qudt/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX unit: <http://qudt.org/vocab/unit/>

SELECT ?samp_pt ?obs_date ?pfastype ?num_val ?unit ?cousub_label
WHERE {{
    ?samp_pt rdf:type me_egad:EGAD-SamplePoint ;
             me_egad:samplePointType me_egad:featureType.PWSW ;
             kwg-ont:sfWithin ?cousub .
    ?obs coso:observedAtSamplePoint ?samp_pt ;
         coso:ofSubstance ?substance ;
         sosa:hasResult ?result ;
         coso:observedTime ?obs_date .
    ?result qudt:quantityValue ?quant_val .
    ?quant_val qudt:numericValue ?num_val ;
               qudt:hasUnit ?unit .
    VALUES ?samp_pt {{ me_egad_data:{sample_point} }}
    BIND(REPLACE(str(?substance), "(.*)\\\\.(.*)", "$2") AS ?pfastype)

    ?cousub rdf:type kwg-ont:AdministrativeRegion_3 ;
            rdfs:label ?cousub_label ;
            kwg-ont:administrativePartOf ?county .
    ?county rdfs:label ?county_label ;
            kwg-ont:administrativePartOf ?state .
    ?state rdfs:label ?state_label .
}}
"""
sparql.setQuery(samples_query)
samples_result = sparql.query()
samples = convertToDataframe(samples_result)
if result.empty:
    raise Exception("No results returned for chosen input parameters.")
print(f'\nMean test results by date and PFAS type for {sample_point}\n')
samples.pivot_table('num_val', index='obs_date', columns='pfastype', aggfunc='mean').fillna('--')


Mean test results by date and PFAS type for samplePoint.152643



KeyError: 'num_val'